In [1]:
# @title **Stress Test: Luau Dataset Preparation (100% Saturated 2048 Sequences)**

# @markdown ---
# @markdown ### **GitHub Repository (Source Code):**
GITHUB_REPO_URL = "https://github.com/bananamort/luau-qwen2.5-coder-1.5b-distillation.git" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}

# @markdown ### **Hugging Face Hub & IO:**
HF_TOKEN = "" # @param {type:"string"}
RAW_DATASET_ID = "TorpedoSoftware/the-luau-stack" # @param {type:"string"}
UPLOAD_DATASET_REPO_ID = "bananamort/the-luau-stack-fim-tokenized-stress" # @param {type:"string"}
OUTPUT_PARQUET_PATH = "stress_fim_train.parquet" # @param {type:"string"}

# @markdown ### **Preprocessing Parameters:**
MAX_SEQ_LEN = 2048 # @param {type:"integer"}
CUTS_PER_FILE = 2 # @param {type:"integer"}
MAX_SAMPLES = 600 # @param {type:"integer"} # 600 full 2048-token saturated sequences

# @markdown ### **Runtime Management:**
AUTO_DISCONNECT_VM = True # @param {type:"boolean"}

import os
import subprocess

try:
    # 1. Git Setup
    if os.path.exists("repo"):
        %cd repo
        !git pull origin {BRANCH}
    elif os.path.exists(".git"):
        !git pull origin {BRANCH}
    elif GITHUB_REPO_URL.strip():
        print(f"Cloning repository: {GITHUB_REPO_URL} (branch: {BRANCH})...")
        !git clone --depth 1 -b {BRANCH} {GITHUB_REPO_URL.strip()} repo
        %cd repo

    # 2. Install Dependencies
    print("Installing dataset preparation dependencies...")
    !pip install -q datasets huggingface_hub pyarrow transformers tokenizers zstandard

    # 3. Execute Preprocessing Pipeline
    cmd = [
        "python", "-u", "src/prep_stress.py",
        "--dataset_id", RAW_DATASET_ID.strip(),
        "--output_parquet", OUTPUT_PARQUET_PATH.strip(),
        "--max_seq_len", str(MAX_SEQ_LEN),
        "--cuts_per_file", str(CUTS_PER_FILE),
    ]
    if MAX_SAMPLES > 0:
        cmd.extend(["--max_samples", str(LIMIT)])
    if HF_TOKEN.strip():
        cmd.extend(["--token", HF_TOKEN.strip()])

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in iter(proc.stdout.readline, ""):
        print(line, end="", flush=True)
    proc.stdout.close()
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

except Exception as e:
    print(f"\nERROR: {e}")
    raise
finally:
    if AUTO_DISCONNECT_VM:
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception:
            pass
